# 84 · Nhánh C seed 43 — **BẢN THỬ NHANH Phần 1** (`r84x_C_s43_thu`)

Mã: gói `nhanh_c` trên GitHub (nhánh `nhanh-c`, commit ghim `2da38a76e91e`); mỗi ô gọi `nhanh_c.buoc(...)` — mã bước = ô đã kiểm của lượt gói ô cắt s42, khác đúng các dòng theo seed (`docs/SPEC_SAU_78Y.md` §B.2ab).

Khác bản chính: `thu=True` (2.048 ảnh train, 1 epoch, 1.024 ảnh val; không ghép cặp, không cổng sức khoẻ). Mục đích: kiểm gói chạy đúng trên Colab, tốc độ / VRAM.

**Chạy:** GPU **G4**, Restart session → **Run all** (~40 phút, ước). **Gửi Claude:** ô tải gói, dòng `cong o cat` (`C4`), các dòng `buoc … VRAM dinh … img/s` + dòng epoch + `O CAT` (`C6`).


## Tải gói `nhanh_c` (GitHub, commit ghim) — chạy trước mọi ô


In [ ]:
# ==== TAI GOI nhanh_c tu GitHub (nhanh mo coi nhanh-c, GHIM commit) - chay MOI phien, TRUOC moi o khac ====
import os, subprocess, sys
NHANH_C_SHA = '2da38a76e91e64335cbde83dd82e507c388e16dd'
os.environ['CXRLT_GOC_ID'] = '1PqpiEocTXJNspAK4GM91bL7JoEaL1Bh_'   # ID thu muc cxrlt_2024 - CHI trong notebook (goi cong khai tren GitHub KHONG chua)
_d_goi = '/content/nhanh_c_goi'
_tep_goi = os.path.normcase(os.path.realpath(os.path.join(_d_goi, 'nhanh_c', '__init__.py')))
_goi_cu = sys.modules.get('nhanh_c')
if _goi_cu is not None:
    if os.path.normcase(os.path.realpath(getattr(_goi_cu, '__file__', ''))) != _tep_goi:
        raise RuntimeError('nhanh_c da nap tu duong dan khac -> Restart session truoc khi tai goi')
    if getattr(_goi_cu, '_NHANH_C_SHA_DA_NAP', None) != NHANH_C_SHA:
        raise RuntimeError('nhanh_c da nap tu commit khac hoac chua duoc loader ghim -> Restart session truoc khi tai goi')
if not os.path.isdir(os.path.join(_d_goi, '.git')):
    subprocess.run(['git', 'clone', '--quiet', '--branch', 'nhanh-c', 'https://github.com/AIVIETNAM-AIO-Tuan/TransferLearning-GPUtask.git', _d_goi], check=True)
subprocess.run(['git', '-C', _d_goi, 'fetch', '--quiet', 'origin', 'nhanh-c'], check=True)
subprocess.run(['git', '-C', _d_goi, 'checkout', '--quiet', '--detach', NHANH_C_SHA], check=True)
_head_goi = subprocess.run(['git', '-C', _d_goi, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert _head_goi == NHANH_C_SHA, f'goi nhanh_c: HEAD {_head_goi} KHAC commit ghim {NHANH_C_SHA} -> DUNG'
if _d_goi not in sys.path: sys.path.insert(0, _d_goi)
import nhanh_c as NC
if os.path.normcase(os.path.realpath(getattr(NC, '__file__', ''))) != _tep_goi:
    raise RuntimeError('nhanh_c import tu duong dan khac goi vua tai -> Restart session')
assert NC.PHIEN_BAN == 'nhanh_c_v1', NC.PHIEN_BAN
NC._NHANH_C_SHA_DA_NAP = NHANH_C_SHA
print(f'nhanh_c {NC.PHIEN_BAN} @ {NHANH_C_SHA[:12]} | {len(NC.ds_buoc())} buoc | seed 43' + (' (BAN THU)' if True else ''))
del _d_goi, _head_goi, _tep_goi, _goi_cu


## Phần 1 · C0 · Cấu hình + dữ liệu + khối

Như J v2; thêm cấu hình `jca_v3.json` và khối `khoi_jca_v3` nhúng nguyên văn. J v3b: thêm cấu hình `jca_v3b.json` và khối `khoi_jca_v3b` (trần write-back) nhúng nguyên văn. Đợt 4: khối ghi Drive một lần (`khoi_ghi_cuoi`); cấu hình `jca_v3b_d4.json` (+ khối `khoi_on_dinh_ft` ở B.4) nhúng nguyên văn; lượt của notebook `BIEN_THE_D4`.

Ô cắt: cấu hình `O_CFG` (5 ô lưới phóng từ ảnh 518, khúc, adapter r 64, skip block 7 → 8, ngưỡng sàng lọc) + đoạn nguyên văn `khoi_cat_vung` / `khoi_cat_sau` + khối `khoi_fusion_study`, `khoi_cat_j`. Đầu ô: chặn rác output tại gốc (httplib2, HF, cảnh báo `requires_grad`; không đổi kết quả); AP mức study bằng `ap_cols` (nhãn không có study dương = NaN thay vì 0).

Nhánh C: cấu hình theo seed (`CAU_HINH_C` do `nhanh_c.buoc` đặt): `SEED`, `RUN_TAG` `r84x_C_s<seed>`, thư mục `stage4_nb84`; khởi động luồng + λ vẫn chép J v3b s42 (như lượt gói s42).


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C0', globals(), seed=43, thu=True)


## Phần 1 · C1 · Kho ảnh 518 cục bộ (nguyên `78u`)

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C1', globals(), seed=43, thu=True)


## Phần 1 · C2 · RAD-DINO + biến đổi + dataset + đầu đọc `38a` (nguyên `78u`) + `fwd_luong` (79a) + `fwd_luong_v2` / `dau_v2` / `loader_study` (78x) — J v3: `dau_v2` thêm đầu nguồn (logsumexp) và kênh vùng cố định

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C2', globals(), seed=43, thu=True)


## Phần 1 · C3 · Bảng lúc train (`78u` + `79a`) + dữ liệu J v2

Neo theo view, ngữ nghĩa vùng của study (ý A), tập vùng S → target ý 2 (cây cha–con), 31 nhãn MIL. In độ phủ vị trí theo tier.

J v3: vùng cố định làm target dự phòng (14 nhãn), loss nhãn-vùng 33 nhãn; λ ×3,23; bảng nguồn nhãn từ `ngu_canh_train` (`78s`, chỉ train).

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C3', globals(), seed=43, thu=True)


## Phần 1 · C4 · Module loss phụ + luồng vùng (neo theo mẫu) + DOCV / DAUV + cổng

Cổng: `fwd_luong_v2 = fwd_ft` lúc khởi tạo (write-back 0); `dau_h_v2` không token vùng = `dau_h`; MIL `w = 0` → logit = đầu đọc ý 1; lô kiểm có cặp frontal–lateral cùng study → mọi phần loss có mặt.

J v3: đầu nguồn + kênh vùng cố định (khởi tạo 0); write-back × 5; CE nguồn chỉ vào đầu nguồn. J v3b: gắn trần write-back theo ảnh (`ap_tran_ghi`) **sau** ×5. Đợt 3 (nhánh khe): tạo nhánh khe (hạt riêng), `ap_khe` sau trần; loss khe vào `PHAN_CFG`.

Ô cắt: module ô (adapter ×2, phép chiếu skip, bộ gộp 6 view) + `fwd_o` / `dau_v2_o`. Cổng: skip lúc đầu → đường ảnh toàn cảnh như B.4 v2; adapter = đồng nhất; 6 view giống nhau = đầu đọc gốc; trường nhìn ô; dòng ngẫu nhiên không đổi; dự đoán tất định (hai lần trùng từng bit — cổng đường chấm `C6` so từng bit); gộp 6 view dạng dày = `GopStudy`.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4', globals(), seed=43, thu=True)


## Phần 1 · C4a · Lọc nhiễu khi tắt worker DataLoader (Python 3.13) — chạy trước `C4b`

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4A', globals(), seed=43, thu=True)


## Phần 1 · C4b · Khởi động luồng vùng (+ head nhãn-vùng)

Backbone, đầu đọc, tháp đông cứng; chỉ luồng + `PHU_V` + head nhãn-vùng học (1 epoch). Ý B không ở đây (cần logit đầu đọc).

Đợt 3: **chép** `luong_khoi_dong.pt` của J v3b s42; nhánh khe: **chép** `khe_khoi_dong_<nhánh>.pt` của notebook chuẩn bị.

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C4B', globals(), seed=43, thu=True)


## Phần 1 · C5 · Tệp resume (nếu có) + λ **chép** từ J v3b s42

Không hiệu chỉnh lại; kiểm λ đang dùng trùng s42 từng số.

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C5', globals(), seed=43, thu=True)


## Phần 1 · C6 · Huấn luyện 6 epoch — ghi Drive một lần

Mỗi epoch: `p_`, nhật ký, chẩn đoán giữ riêng, `rs_` nhẹ, tệp resume ghi đè; `backbone_ft.pt` (epoch 6) thêm `cfg_d4`. B.4: EMA sau mỗi bước, VAL EMA + thô.

Ô cắt: train / dự đoán / chẩn đoán giữ riêng qua `fwd_o` + `dau_v2_o`; EMA thêm 4 module ô; `rs_` / `backbone_ft.pt` / tệp resume thêm module ô + `cfg_o`; dòng `O CAT` mỗi epoch.

Nhánh C: cổng ghép cặp dấu vân lô chỉ ở s42; luật thử seed 44 của J v3b tắt.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C6', globals(), seed=43, thu=True)


## Phần 1 · C7 · Bảng lượt này cạnh J v3b s42 + dấu vân lô + λ + write-back / trần — báo kèm

In [ ]:
import nhanh_c as NC; NC.buoc('p1_C7', globals(), seed=43, thu=True)


## Phần 1 · C9 · `dong_bo()` (ghi một lần) — chạy trước khi tắt runtime; kiểm `rs_` nhẹ ep1–6 + `backbone_ft.pt`, tệp resume đã xoá

Nhánh C: **cổng sức khoẻ** — VAL trong lượt TB ep1–6 thấp hơn C s42 (0,2619) quá 0,01 → `YEU`, ô chuyển phần dừng (báo Claude); Tuấn quyết chạy tiếp: chạy ô `TIEP_TUC_C = True` rồi chạy lại ô chuyển.


In [ ]:
import nhanh_c as NC; NC.buoc('p1_C9', globals(), seed=43, thu=True)
